# Notebook 05 — Simulation Engine & What-If Scenarios
**Prerequisites:** `models/checkpoints/lstm_best.pt` + `data/raw/ADNIMERGE.csv`

In [ ]:
# ── Cell 1: Imports (VS Code / Jupyter compatible) ───────────────────
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import os, warnings, json
warnings.filterwarnings('ignore')

# FIX: Do NOT use matplotlib.use('Agg') in VS Code Jupyter
# Just import directly — VS Code handles the backend automatically
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split

np.random.seed(42)
torch.manual_seed(42)
os.makedirs('results/visualizations', exist_ok=True)

DEVICE = torch.device('cpu')
BG     = '#0d1117'
PANEL  = '#161b22'
FG     = 'white'
MUTED  = '#8b949e'
GRID   = '#21262d'

def style_ax(ax):
    ax.set_facecolor(PANEL)
    ax.tick_params(colors=FG, labelsize=9)
    ax.xaxis.label.set_color(FG)
    ax.yaxis.label.set_color(FG)
    ax.title.set_color(FG)
    for sp in ['bottom','left']:  ax.spines[sp].set_color(GRID)
    for sp in ['top','right']:    ax.spines[sp].set_visible(False)
    ax.grid(color=GRID, alpha=0.5, linewidth=0.5)

from sklearn.linear_model import LinearRegression
print('Imports OK')

In [ ]:
# ── Cell 2: Load data & rebuild scalers (matches NB02 exactly) ──────
df = pd.read_csv('data/raw/ADNIMERGE.csv')
df['Hippocampus'] = df['Hippocampus'].fillna(df['Hippocampus'].mean())
df['APOE4']       = df['APOE4'].fillna(df['APOE4'].median())
df['Education']   = df['Education'].fillna(df['Education'].median())
df = df.dropna(subset=['MMSE', 'visit_num']).sort_values(['RID','visit_num']).reset_index(drop=True)

# ── Must exactly match NB02 ──────────────────────────────────────────
FEAT_COLS = ['MMSE', 'Hippocampus', 'APOE4', 'Education', 'visit_num']
N_FEAT    = len(FEAT_COLS)   # 5
MAX_LEN   = 8

# Rebuild scaler on all data (same as NB02 Cell 4)
for col in FEAT_COLS:
    if col in df.columns:
        df[col] = df[col].fillna(df[col].median())

feat_scaler  = StandardScaler()
feat_scaler.fit(df[FEAT_COLS].values.astype('float32'))

# Target stats for denormalising MMSE predictions (same as NB02)
target_mean = float(df['MMSE'].mean())
target_std  = float(df['MMSE'].std())

# Hippocampus: fit a simple linear slope from MMSE so we can estimate it
# from predicted MMSE (used only for visualization — not loaded from checkpoint)
from sklearn.linear_model import LinearRegression
sub = df.dropna(subset=['MMSE','Hippocampus'])
hippo_lr = LinearRegression().fit(sub[['MMSE']], sub['Hippocampus'])

print(f'Features ({N_FEAT}): {FEAT_COLS}')
print(f'target_mean={target_mean:.2f}  target_std={target_std:.2f}')
print(f'Hippo estimator slope: {hippo_lr.coef_[0]:.1f} mm³/MMSE-pt')


In [ ]:
# ── Cell 3: Rebuild AttentionLSTM — must match NB02 Cell 6 exactly ──
class AttentionLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=128, num_layers=2,
                 dropout=0.3, bidirectional=True):
        super().__init__()
        self.bidirectional = bidirectional
        D = 2 if bidirectional else 1
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers,
                            batch_first=True, dropout=dropout,
                            bidirectional=bidirectional)
        self.attn = nn.Linear(hidden_size * D, 1)
        self.fc   = nn.Sequential(
            nn.Linear(hidden_size * D + 1, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 1)
        )

    def forward(self, x, mask, last_mmse):
        out, _  = self.lstm(x)                  # (B, T, hidden*D)
        scores  = self.attn(out).squeeze(-1)    # (B, T)
        scores  = scores.masked_fill(mask == 0, -1e9)
        weights = torch.softmax(scores, dim=1).unsqueeze(-1)
        ctx     = (out * weights).sum(dim=1)    # (B, hidden*D)
        last_n  = last_mmse if last_mmse.dim() == 2 else last_mmse.unsqueeze(-1)
        ctx     = torch.cat([ctx, last_n], dim=-1)
        return self.fc(ctx)                     # (B, 1) — MMSE only


CKPT = 'models/checkpoints/lstm_best.pt'
state = torch.load(CKPT, map_location=DEVICE)

# Verify checkpoint matches expected architecture
ckpt_in  = state['lstm.weight_ih_l0'].shape[1]
ckpt_hid = state['lstm.weight_ih_l0'].shape[0] // 4
print(f'Checkpoint: input={ckpt_in}, hidden={ckpt_hid}')
assert ckpt_in == N_FEAT, f'Feature mismatch: checkpoint={ckpt_in}, FEAT_COLS={N_FEAT}'

model = AttentionLSTM(input_size=N_FEAT, hidden_size=128,
                      num_layers=2, bidirectional=True).to(DEVICE)
model.load_state_dict(state)
model.eval()
print('Model loaded OK — exact match.')


In [ ]:
# ── Cell 4: Prediction helpers (updated for AttentionLSTM) ──────────
def predict_next(seq_raw):
    """
    seq_raw: np.array shape (T, 5) — raw (unscaled) feature rows.
    Returns: (mmse_pred, hippo_pred) both in original units.
    """
    seq_s = feat_scaler.transform(seq_raw.astype('float32'))
    T = seq_s.shape[0]
    use = min(T, MAX_LEN)
    pad = MAX_LEN - use
    padded       = np.zeros((MAX_LEN, N_FEAT), dtype='float32')
    mask         = np.zeros(MAX_LEN, dtype='float32')
    padded[pad:] = seq_s[-use:]
    mask[pad:]   = 1.0

    # last_mmse: normalised scalar needed by AttentionLSTM.forward()
    raw_last_mmse = float(seq_raw[-1, FEAT_COLS.index('MMSE')])
    last_mmse_n   = (raw_last_mmse - target_mean) / target_std

    x_t  = torch.tensor(padded).unsqueeze(0)                     # (1, MAX_LEN, 5)
    m_t  = torch.tensor(mask).unsqueeze(0)                       # (1, MAX_LEN)
    lm_t = torch.tensor([[last_mmse_n]], dtype=torch.float32)    # (1, 1)

    with torch.no_grad():
        out = model(x_t, m_t, lm_t)          # (1, 1) — normalised MMSE

    mmse_p  = float(np.clip(out.item() * target_std + target_mean, 0, 30))
    hippo_p = float(np.clip(hippo_lr.predict([[mmse_p]])[0], 3000, 9000))
    return mmse_p, hippo_p


def build_next_row(prev_raw, new_mmse, new_hippo, baseline_mmse, visit_idx):
    """Build the next feature row (5 features matching FEAT_COLS)."""
    nr = prev_raw.copy()
    nr[FEAT_COLS.index('MMSE')]        = new_mmse
    nr[FEAT_COLS.index('Hippocampus')] = new_hippo
    nr[FEAT_COLS.index('visit_num')]   = prev_raw[FEAT_COLS.index('visit_num')] + 1
    # APOE4 and Education stay constant across visits
    return nr


def predict_trajectory(context_raw, n_future=6, mmse_scale=1.0):
    """
    Auto-regressively predict n_future visits.
    mmse_scale < 1.0 simulates intervention (reduces predicted decline).
    Returns: (mmse_preds, hippo_preds) as lists of length n_future.
    """
    seq           = context_raw.copy().astype('float32')
    baseline_mmse = float(seq[0, FEAT_COLS.index('MMSE')])
    last_mmse     = float(seq[-1, FEAT_COLS.index('MMSE')])
    mmse_preds, hippo_preds = [], []

    for i in range(n_future):
        raw_mmse, raw_hippo = predict_next(seq)

        # Apply intervention scaling to the *decline* only
        if mmse_scale != 1.0:
            decline  = last_mmse - raw_mmse
            raw_mmse = float(np.clip(last_mmse - decline * mmse_scale, 0, 30))
            raw_hippo = float(np.clip(hippo_lr.predict([[raw_mmse]])[0], 3000, 9000))

        mmse_preds.append(raw_mmse)
        hippo_preds.append(raw_hippo)
        last_mmse = raw_mmse

        next_row = build_next_row(seq[-1], raw_mmse, raw_hippo, baseline_mmse, len(seq)+i)
        seq      = np.vstack([seq, next_row])

    return mmse_preds, hippo_preds


# Quick sanity check
test_pat = df[df['RID']==750].dropna(subset=['MMSE','visit_num']).sort_values('visit_num')
test_ctx = test_pat[FEAT_COLS].values.astype('float32')
tm, th   = predict_next(test_ctx)
print(f'Sanity check RID 750: predicted MMSE={tm:.1f}  Hippo={th:.0f} mm³')
print('predict_next OK — Cell 5 onwards should run cleanly.')


In [ ]:
# ── Cell 5: Demo patient setup ────────────────────────────────────────
DEMO_RIDS   = [750, 667, 1282, 50, 128]
DEMO_COLORS = ['#ef5350','#f59e0b','#ab47bc','#4fc3f7','#ff7043']
N_FUTURE    = 6

demo_data = {}
for rid in DEMO_RIDS:
    pat   = df[df['RID']==rid].dropna(subset=['MMSE','visit_num']).sort_values('visit_num').reset_index(drop=True)
    ctx   = pat[FEAT_COLS].values.astype(np.float32)
    visits = pat['visit_num'].values
    mmse   = pat['MMSE'].values
    hippo  = pat['Hippocampus'].values
    apoe   = int(pat['APOE4'].dropna().iloc[0]) if len(pat['APOE4'].dropna())>0 else 0
    pred_m, pred_h = predict_trajectory(ctx, n_future=N_FUTURE)
    last_v  = visits[-1]
    pred_v  = [last_v + i + 1 for i in range(N_FUTURE)]
    demo_data[rid] = dict(ctx=ctx, visits=visits, mmse=mmse, hippo=hippo,
                          apoe=apoe, pred_m=pred_m, pred_h=pred_h,
                          last_v=last_v, pred_v=pred_v)

print('Demo patients ready:')
for rid in DEMO_RIDS:
    d = demo_data[rid]
    print(f'  RID {rid}  APOE4={d["apoe"]}  visits={len(d["visits"])}  pred[-1]={d["pred_m"][-1]:.1f}')

In [ ]:
# ── Cell 6: APOE4 What-If ─────────────────────────────────────────────
RID_BASE   = 750
d          = demo_data[RID_BASE]
ctx_base   = d['ctx'].copy()
obs_visits = d['visits']
obs_mmse   = d['mmse']
obs_hippo  = d['hippo']
last_v     = d['last_v']
pred_v     = d['pred_v']

APOE_COLORS = {0:'#4fc3f7', 1:'#f59e0b', 2:'#ef5350'}
APOE_LABELS = {0:'APOE4=0 (low risk)', 1:'APOE4=1 (moderate)', 2:'APOE4=2 (high risk)'}

apoe_results = {}
for apoe_val in [0, 1, 2]:
    ctx = ctx_base.copy()
    ctx[:, FEAT_COLS.index('APOE4')] = float(apoe_val)
    apoe_results[apoe_val] = predict_trajectory(ctx, n_future=N_FUTURE)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

for ax_idx, (ax, ylabel, key) in enumerate(zip(
    axes, ['MMSE Score', 'Hippocampus Volume (mm³)'], ['mmse', 'hippo']
)):
    style_ax(ax)
    obs_vals = obs_mmse if key=='mmse' else obs_hippo
    ax.plot(obs_visits, obs_vals, color=FG, lw=2.5, marker='o', ms=6,
            label='Observed (RID 750)', zorder=5)
    for apoe_val in [0, 1, 2]:
        pred_m, pred_h = apoe_results[apoe_val]
        preds = pred_m if key=='mmse' else pred_h
        ax.plot([last_v]+pred_v, [obs_vals[-1]]+list(preds),
                color=APOE_COLORS[apoe_val], lw=2, ls='--', marker='s', ms=5,
                label=APOE_LABELS[apoe_val])
    if key == 'mmse':
        ax.axhline(24, color=MUTED, ls=':', lw=1)
        ax.text(obs_visits[0], 24.6, 'MCI threshold (24)', color=MUTED, fontsize=8)
        ax.set_ylim(0, 32)
    ax.set_title(f'APOE4 What-If — {ylabel}', fontsize=11)
    ax.set_xlabel('Visit Number', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(facecolor=PANEL, labelcolor=FG, fontsize=8)

mmse_spread = apoe_results[0][0][-1] - apoe_results[2][0][-1]
print(f'MMSE spread APOE4=0 vs APOE4=2 at visit {pred_v[-1]}: {mmse_spread:.2f} pts')

fig.suptitle('What-If Simulation: APOE4 Genetic Risk Scenarios',
             color=FG, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/visualizations/what_if_apoe4.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → results/visualizations/what_if_apoe4.png')

In [ ]:
# ── Cell 7: Monte Carlo Confidence Bands ──────────────────────────────
N_SAMPLES = 200
NOISE_STD = 1.0

mc_mmse_trajs, mc_hippo_trajs = [], []
np.random.seed(42)
print(f'Running {N_SAMPLES} Monte Carlo samples...')

for i in range(N_SAMPLES):
    ctx = ctx_base.copy()
    ctx[:, 0] += np.random.normal(0, NOISE_STD, size=len(ctx))
    ctx[:, 0]  = np.clip(ctx[:, 0], 0, 30)
    if N_FEAT > 5:
        for t in range(1, len(ctx)):
            ctx[t, 5] = ctx[t,0] - ctx[t-1,0]
            ctx[t, 6] = ctx[t,0] - ctx[0,0]
            ctx[t, 7] = ctx[t,6] / max(t, 1)
    pred_m, pred_h = predict_trajectory(ctx.astype(np.float32), n_future=N_FUTURE)
    mc_mmse_trajs.append(pred_m)
    mc_hippo_trajs.append(pred_h)
    if (i+1) % 50 == 0:
        print(f'  {i+1}/{N_SAMPLES} done')

mc_mmse  = np.array(mc_mmse_trajs)
mc_hippo = np.array(mc_hippo_trajs)

mmse_mean = mc_mmse.mean(axis=0)
mmse_p5   = np.percentile(mc_mmse,  5, axis=0)
mmse_p25  = np.percentile(mc_mmse, 25, axis=0)
mmse_p75  = np.percentile(mc_mmse, 75, axis=0)
mmse_p95  = np.percentile(mc_mmse, 95, axis=0)
hippo_mean = mc_hippo.mean(axis=0)
hippo_p5   = np.percentile(mc_hippo,  5, axis=0)
hippo_p95  = np.percentile(mc_hippo, 95, axis=0)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

ax = axes[0]
style_ax(ax)
ax.plot(obs_visits, obs_mmse, color=FG, lw=2.5, marker='o', ms=6, label='Observed', zorder=5)
ax.fill_between(pred_v, mmse_p5, mmse_p95, color='#a78bfa', alpha=0.20, label='90% CI')
ax.fill_between(pred_v, mmse_p25, mmse_p75, color='#a78bfa', alpha=0.35, label='50% CI')
ax.plot([last_v]+pred_v, [obs_mmse[-1]]+list(mmse_mean),
        color='#a78bfa', lw=2, ls='--', marker='s', ms=5, label='Mean prediction', zorder=4)
ax.axhline(24, color=MUTED, ls=':', lw=1)
ax.set_ylim(0, 32)
ax.set_title(f'Monte Carlo MMSE (n={N_SAMPLES})', fontsize=11)
ax.set_xlabel('Visit Number', fontsize=9)
ax.set_ylabel('MMSE Score', fontsize=9)
ax.legend(facecolor=PANEL, labelcolor=FG, fontsize=8)

ax = axes[1]
style_ax(ax)
ax.plot(obs_visits, obs_hippo, color=FG, lw=2.5, marker='o', ms=6, label='Observed', zorder=5)
ax.fill_between(pred_v, hippo_p5, hippo_p95, color='#34d399', alpha=0.25, label='90% CI')
ax.plot([last_v]+pred_v, [obs_hippo[-1]]+list(hippo_mean),
        color='#34d399', lw=2, ls='--', marker='s', ms=5, label='Mean prediction', zorder=4)
ax.set_title(f'Monte Carlo Hippocampus (n={N_SAMPLES})', fontsize=11)
ax.set_xlabel('Visit Number', fontsize=9)
ax.set_ylabel('Hippocampus Volume (mm³)', fontsize=9)
ax.legend(facecolor=PANEL, labelcolor=FG, fontsize=8)

ci_width = mmse_p95[-1] - mmse_p5[-1]
fig.suptitle(f'Monte Carlo Uncertainty — RID {RID_BASE}  (σ={NOISE_STD})',
             color=FG, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/visualizations/monte_carlo_ci.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print(f'Saved → results/visualizations/monte_carlo_ci.png')
print(f'At visit {pred_v[-1]}: MMSE={mmse_mean[-1]:.1f} (90% CI: {mmse_p5[-1]:.1f}–{mmse_p95[-1]:.1f})')

In [ ]:
# ── Cell 8: Intervention Scenarios ───────────────────────────────────
scenarios = {
    'No intervention':    (1.0, '#ef5350'),
    '20% decline ↓':      (0.8, '#f59e0b'),
    '40% decline ↓':      (0.6, '#4ade80'),
    'Halt (theoretical)': (0.0, '#4fc3f7'),
}

int_results = {}
for label, (scale, color) in scenarios.items():
    pred_m, pred_h = predict_trajectory(ctx_base.copy(), n_future=N_FUTURE, mmse_scale=scale)
    int_results[label] = (pred_m, pred_h, color)

fig, axes = plt.subplots(1, 2, figsize=(15, 5))
fig.patch.set_facecolor(BG)

for ax, ylabel, key in zip(axes, ['MMSE Score','Hippocampus (mm³)'], ['mmse','hippo']):
    style_ax(ax)
    obs_vals = obs_mmse if key=='mmse' else obs_hippo
    ax.plot(obs_visits, obs_vals, color=FG, lw=2.5, marker='o', ms=6,
            label='Observed', zorder=5)
    for label, (pred_m, pred_h, color) in int_results.items():
        preds = pred_m if key=='mmse' else pred_h
        ls = '-' if 'Halt' in label else '--'
        ax.plot([last_v]+pred_v, [obs_vals[-1]]+list(preds),
                color=color, lw=2, ls=ls, marker='s', ms=4, label=label)
    if key == 'mmse':
        ax.axhline(24, color=MUTED, ls=':', lw=1)
        ax.set_ylim(0, 32)
        nat_m   = int_results['No intervention'][0]
        int40_m = int_results['40% decline ↓'][0]
        ax.fill_between(pred_v, nat_m, int40_m, color='#4ade80', alpha=0.12)
    ax.set_title(f'Intervention Scenarios — {ylabel}', fontsize=11)
    ax.set_xlabel('Visit Number', fontsize=9)
    ax.set_ylabel(ylabel, fontsize=9)
    ax.legend(facecolor=PANEL, labelcolor=FG, fontsize=8)

benefit = int_results['40% decline ↓'][0][-1] - int_results['No intervention'][0][-1]
print(f'40% intervention benefit at visit {pred_v[-1]}: +{benefit:.1f} MMSE pts')

fig.suptitle(f'Intervention Delay Modelling — RID {RID_BASE}',
             color=FG, fontsize=13, fontweight='bold')
plt.tight_layout()
plt.savefig('results/visualizations/what_if_intervention.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → results/visualizations/what_if_intervention.png')

In [ ]:
# ── Cell 9: All 5 demo patients ───────────────────────────────────────
fig, axes = plt.subplots(1, 5, figsize=(22, 5), sharey=True)
fig.patch.set_facecolor(BG)

for ax, rid, color in zip(axes, DEMO_RIDS, DEMO_COLORS):
    style_ax(ax)
    d = demo_data[rid]
    ax.plot(d['visits'], d['mmse'], 'o-', color=color, lw=2, ms=5,
            label='Observed', zorder=4)
    ax.plot([d['last_v']]+d['pred_v'], [d['mmse'][-1]]+list(d['pred_m']),
            's--', color=color, lw=1.8, ms=4, alpha=0.9, label='Predicted')
    int_m, _ = predict_trajectory(d['ctx'].copy(), n_future=N_FUTURE, mmse_scale=0.6)
    ax.plot([d['last_v']]+d['pred_v'], [d['mmse'][-1]]+list(int_m),
            '^:', color='#4ade80', lw=1.5, ms=4, alpha=0.85, label='40% slowdown')
    ax.fill_between(d['pred_v'], d['pred_m'], int_m, color='#4ade80', alpha=0.12)
    ax.axhline(24, color=MUTED, ls=':', lw=0.8, alpha=0.7)
    ax.set_ylim(0, 32)
    ax.set_xlabel('Visit', fontsize=8)
    ax.set_title(f'RID {rid}  APOE4={d["apoe"]}', color=FG, fontsize=9)

axes[0].set_ylabel('MMSE Score', color=FG, fontsize=9)
axes[0].legend(facecolor=PANEL, labelcolor=FG, fontsize=7)
fig.suptitle('Digital Twin — Natural vs Intervention (All Demo Patients)',
             color=FG, fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('results/visualizations/all_patients_simulation.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print('Saved → results/visualizations/all_patients_simulation.png')

In [ ]:
# ── Cell 10: Save summary ─────────────────────────────────────────────
summary = {
    'base_patient':    RID_BASE,
    'n_future_visits': N_FUTURE,
    'apoe4_mmse_spread':   round(float(mmse_spread), 2),
    'mc_samples':          N_SAMPLES,
    'mc_ci90_width_last':  round(float(ci_width), 2),
    'intervention_40pct_benefit': round(float(benefit), 2),
}
with open('results/metrics/simulation_summary.json','w') as f:
    json.dump(summary, f, indent=2)

print('='*50)
print('NOTEBOOK 05 COMPLETE')
print('='*50)
files = [
    'results/visualizations/what_if_apoe4.png',
    'results/visualizations/monte_carlo_ci.png',
    'results/visualizations/what_if_intervention.png',
    'results/visualizations/all_patients_simulation.png',
    'results/metrics/simulation_summary.json',
]
for f in files:
    status = '✓' if os.path.exists(f) else '✗ MISSING'
    print(f'  {status}  {f}')
print()
print(json.dumps(summary, indent=2))